In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import math
from itertools import combinations

data_dir = "../data"
prompts_dir = "../prompts"
predictions_dir = "../predictions"
results_dir = "../results"

In [ ]:
from typing import Iterable, Tuple
from scipy import stats

SIGNIFICANCE_ALPHA = 0.05
USE_BH_CORRECTION = False

# Style template
MODEL_ORDER = [
    'claude-opus-4', 'llama-3.1-405b', 'gpt-4o', 'grok-3', 'gemini-2.0-flash',
    'claude-3.5-haiku', 'llama-3.1-8b', 'gpt-4o-mini', 'grok-3-mini'
]
COLORS = {
    'neg': 'royalblue',
    'pos': 'orange',
    'other': 'lightgray',
}

gg_issue_inputs = [
    'binge-watching', 'drones', 'fighting-in-hockey', 'olympics', 'ronald-reagan',
    'saturday-halloween', 'school-uniforms', 'standardized-tests', 'teacher-tenure', 'us-penny',
]


def apply_paper_style():
    plt.rcParams.update({
        'font.size': 24,
        'axes.titlesize': 28,
        'axes.labelsize': 24,
        'xtick.labelsize': 22,
        'ytick.labelsize': 22,
        'legend.fontsize': 20,
    })


def bootstrap_ci(values: Iterable[float], n_boot: int = 2000, ci: float = 0.95, seed: int = 42) -> Tuple[float, float]:
    arr = np.asarray(list(values), dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return (np.nan, np.nan)
    if len(arr) == 1:
        return (arr[0], arr[0])

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        sample = rng.choice(arr, size=len(arr), replace=True)
        boots[i] = np.mean(sample)

    alpha = 1 - ci
    lower = np.quantile(boots, alpha / 2)
    upper = np.quantile(boots, 1 - alpha / 2)
    return (float(lower), float(upper))


def p_to_marker(p: float, alpha: float = SIGNIFICANCE_ALPHA) -> str:
    if np.isnan(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < alpha:
        return "*"
    return "ns"


def apply_bh_correction(p_values: Iterable[float]) -> np.ndarray:
    """Benjamini-Hochberg correction with no extra dependency."""
    p = np.asarray(list(p_values), dtype=float)
    n = len(p)
    if n == 0:
        return p
    order = np.argsort(p)
    ranked = p[order]
    adjusted = ranked * n / (np.arange(1, n + 1))
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty_like(adjusted)
    out[order] = adjusted
    return out


def add_ci_columns(df: pd.DataFrame, value_col: str, group_cols: list[str], out_prefix: str) -> pd.DataFrame:
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        mean_v = float(np.mean(g[value_col]))
        ci_low, ci_high = bootstrap_ci(g[value_col].values)
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: val for col, val in zip(group_cols, keys)}
        row[f"{out_prefix}_mean"] = mean_v
        row[f"{out_prefix}_ci_low"] = ci_low
        row[f"{out_prefix}_ci_high"] = ci_high
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
# Helper smoke checks
assert p_to_marker(0.2) == "ns"
assert p_to_marker(0.01) in {"*", "**"}

lo, hi = bootstrap_ci([0.1, 0.2, 0.3, 0.4], n_boot=200, seed=1)
assert lo <= hi

bh = apply_bh_correction([0.001, 0.02, 0.2])
assert len(bh) == 3
print("CI/significance helpers ready.")

In [ ]:
high_level_topic_map = {
    'Digital Life, Science, & Technology': [
        'pokemon-go',
        'artificial-intelligence-AI',
        'binge-watching',
        'cell-phones',
        'internet',
        'net-neutrality',
        'social-media',
        'space-colonization',
        'tiktok',
        'video-games',
        'tablets-vs-textbooks'
    ], 'Government & Civics': [
        'american-socialism',
        'congressional-term-limits',
        'dc-and-puerto-rico-statehood',
        'filibuster',
        'mandatory-national-service',
        'sanctuary-cities',
        'us-supreme-court',
        'american-civil-liberties-union-aclu',
        'us-pledge-of-allegiance'
    ], 'Education': [
        'animal-dissection',
        'book-bans',
        'cell-phones',
        'college-education',
        'corporal-punishment',
        'free-college',
        'homework',
        'school-uniforms',
        'school-vaccine-mandates',
        'standardized-tests',
        'student-loan-debt',
        'school-vouchers',
        'teacher-tenure',
    ], 'Health & Medicine': [
        'abortion',
        'birth-control',
        'maid-medical-aid-in-dying',
        'medical-marijuana',
        'milk',
        'obesity',
        'prescription-drug-costs',
        'universal-health-care',
        'vaping',
        'employer-vaccine-mandates',
        'obamacare',
    ], 'Environment & Animal Rights': [
        'alternative-energy',
        'animal-dissection',
        'animal-testing',
        'cannabidiol-cbd-for-pets',
        'fracking',
        'fur-clothing-bans',
        'gmos',
        'pit-bull-bans',
        'single-use-plastics',
        'vegetarianism',
        'zoos',
        'climate-change',
        'dakota-access-pipeline',
        'fur-clothing-bans',
        'gmos',
        'pit-bull-bans',
        'single-use-plastics',
        'vegetarianism',
        'zoos',
        'climate-change',
        'dakota-access-pipeline'
    ], 'Elections & Presidents': [
        'election-day',
        'electoral-college',
        'felon-voting',
        'voting-age',
        'voting-machines',
        'bill-clinton',
        'ronald-reagan',
    ], 'Society & Holidays': [
        'cancel-culture',
        'daylight-saving-time',
        'dress-codes',
        'historical-statue-removal',
        'homelessness',
        'reparations-for-slavery',
        'sanctuary-cities',
        'santa-claus',
        'saturday-halloween',
        'gay-marriage',
        'ride-sharing',
    ], 'Sports': [
        'cheerleading',
        'fighting-in-hockey',
        'football',
        'golf',
        'national-anthem-protest',
        'olympics',
        'sports-and-drugs',
        'college-football-playoffs',
        'paying-college-athletes',
    ], 'Law & Order': [
        'constitutional-carry-of-guns',
        'death-penalty',
        'defund-the-police',
        'drinking-age',
        'gun-control',
        'police-body-cameras',
        'private-prisons',
        'recreational-marijuana-legalization',
        'dare-drug-abuse-resistance-education',
        'prostitution',
    ], 'Economy & Taxes': [
        'corporate-tax-rate',
        'gig-economy',
        'gold-standard',
        'minimum-wage',
        'social-security',
        'universal-basic-income-ubi',
        'us-penny',
        'big-three-auto',
        'churches-and-taxes',
        'insider-trading-by-congress',
    ], 'Immigration & International Scene': [
        'cuba-embargo',
        'immigration',
        'sanctuary-cities',
        'drones',
        'us-drone-shot-down-by-iran',
        'us-iraq-war',
        'wtc-muslim-center',
    ]
}

topic_high_level_map = {}
for key, value in high_level_topic_map.items():
    for v in value:
        topic_high_level_map[v] = key

## Paper Figure Regeneration with CI and Significance

This section regenerates final paper figures in `figures/` with explicit confidence intervals and significance annotations.

In [ ]:
import glob

stats_df = pd.read_csv('../results/statistical_summary.csv')
order_df = pd.read_csv('../results/order_effects.csv')

model_result_files = sorted(glob.glob('../results/*.csv'))
model_frames = []
for p in model_result_files:
    name = os.path.basename(p)
    if name in {'statistical_summary.csv', 'order_effects.csv'}:
        continue
    try:
        df_ = pd.read_csv(p)
    except Exception:
        continue
    if {'issue', 'issue_stance', 'count', 'case', 'model'}.issubset(df_.columns):
        model_frames.append(df_)

results_long = pd.concat(model_frames, ignore_index=True)
results_long['model'] = results_long['model'].str.replace('.csv', '', regex=False)

# Build OM source data directly so this notebook runs standalone.
# Mirror exploratory notebook logic so OM scores are on the same scale.

def calculate_open_mindedness_scores(pivot_df, models, case_weights):
    baseline_case = 'neither'
    open_mindedness_data = []

    for model in models:
        if model not in pivot_df.columns:
            continue

        for issue in pivot_df['issue'].unique():
            issue_data = pivot_df[pivot_df['issue'] == issue]
            if issue_data.empty:
                continue

            baseline_data = issue_data[issue_data['case'] == baseline_case]
            if baseline_data.empty:
                continue

            baseline_stances = baseline_data[['issue_stance', model]].copy().sort_values(model, ascending=False)
            if baseline_stances.empty:
                continue

            # If top stance is "other", use next most likely non-other stance when available.
            if baseline_stances.iloc[0]['issue_stance'] == 'other' and len(baseline_stances) > 1:
                baseline_stance = baseline_stances.iloc[1]['issue_stance']
                baseline_share = baseline_stances.iloc[1][model]
            else:
                baseline_stance = baseline_stances.iloc[0]['issue_stance']
                baseline_share = baseline_stances.iloc[0][model]

            issue_score = 0.0
            stance_flips = []

            for case in issue_data['case'].unique():
                if case == baseline_case:
                    continue

                case_data = issue_data[issue_data['case'] == case]
                if case_data.empty:
                    continue

                case_stances = case_data[['issue_stance', model]].copy().sort_values(model, ascending=False)
                if case_stances.empty:
                    continue

                case_stance = case_stances.iloc[0]['issue_stance']
                case_share = case_stances.iloc[0][model]

                if case_stance != baseline_stance:
                    magnitude = abs(float(case_share) - float(baseline_share))
                    case_weight = case_weights.get(case, 1)
                    weighted_score = case_weight * magnitude
                    issue_score += weighted_score

                    stance_flips.append({
                        'case': case,
                        'from_stance': baseline_stance,
                        'to_stance': case_stance,
                        'magnitude': magnitude,
                        'weight': case_weight,
                        'weighted_score': weighted_score,
                    })

            open_mindedness_data.append({
                'model': model,
                'issue': issue,
                'baseline_stance': baseline_stance,
                'baseline_share': baseline_share,
                'open_mindedness_score': issue_score,
                'num_stance_flips': len(stance_flips),
                'stance_flips': stance_flips,
            })

    out_df = pd.DataFrame(open_mindedness_data)
    if not out_df.empty:
        # Match exploratory normalization into roughly 0-100 range.
        out_df['open_mindedness_score'] = out_df['open_mindedness_score'] / 9 * 100
    return out_df


# Convert long counts into a pivot with one column per model and rows per issue/case/stance.
pivot = results_long.pivot_table(
    index=['issue', 'case', 'issue_stance'],
    columns='model',
    values='count',
    aggfunc='sum',
    fill_value=0,
).reset_index()

# Normalize each model by total responses per issue/case to get shares.
model_cols = [c for c in pivot.columns if c not in {'issue', 'case', 'issue_stance'}]
for m in model_cols:
    denom = pivot.groupby(['issue', 'case'])[m].transform('sum')
    pivot[m] = (pivot[m] / denom.replace(0, np.nan)).fillna(0.0)

case_weights = {'pro': 1, 'con': 1, '75pro': 2, '75con': 2, 'all': 3, 'neither': 0}
full_run_models = sorted(model_cols)

open_mindedness_df = calculate_open_mindedness_scores(pivot, full_run_models, case_weights)

# Attach high-level topics.
issue_to_topic = {}
for topic, issues in high_level_topic_map.items():
    for issue in issues:
        issue_to_topic[issue] = topic
open_mindedness_df['high_level_topic'] = open_mindedness_df['issue'].map(issue_to_topic).fillna('Other')

subset_models = set(order_df['model'].unique())
subset_open_mindedness_df = open_mindedness_df[open_mindedness_df['model'].isin(subset_models)].copy()

print('stats rows:', len(stats_df), '| order rows:', len(order_df), '| long rows:', len(results_long), '| om rows:', len(open_mindedness_df))

In [ ]:
# PAPER_ARTIFACT: pro_share_shift_by_evidence_case.pdf
apply_paper_style()

case_order = ['con', '75con', 'all', '75pro', 'pro']
case_labels = ['OS con', 'C&C con', 'Balanced', 'C&C pro', 'OS pro']
plot_df = stats_df[stats_df['case'].isin(case_order)].copy()

# models = [m for m in MODEL_ORDER if m in plot_df['model'].unique()]
models = [
    'claude-opus-4', 'llama-3.1-405b', 'gpt-4o',
    'claude-3.5-haiku', 'llama-3.1-8b', 'gpt-4o-mini',
    'grok-3', 'grok-3-mini', 'gemini-2.0-flash'
]
n_cols = 3
n_rows = int(np.ceil(len(models) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 4.2 * n_rows), sharex=True)
axes = np.array(axes).reshape(-1)

for i, model in enumerate(models):
    ax = axes[i]
    mdf = plot_df[plot_df['model'] == model].set_index('case').reindex(case_order)
    y = mdf['mean_pro_shift'].values
    ci_low = mdf['ci_lower'].values
    ci_high = mdf['ci_upper'].values
    yerr = np.vstack([y - ci_low, ci_high - y])

    colors = [COLORS['neg'] if v < 0 else COLORS['pos'] for v in y]
    ax.barh(case_labels, y, color=colors, alpha=0.95)
    ax.errorbar(y, case_labels, xerr=yerr, fmt='none', ecolor='black', elinewidth=0.8, capsize=2)

    # significance markers from p-values
    # pvals = mdf['p_value'].values
    # for yy, vv, pp in zip(case_labels, y, pvals):
    #     marker = p_to_marker(pp)
    #     if marker and marker != 'ns':
    #         xoff = 0.03 if vv >= 0 else -0.03
    #         ax.text(vv + xoff, yy, marker, va='center', ha='left' if vv >= 0 else 'right', fontsize=16)

    ax.set_title(model)
    ax.axvline(0, color='black', linewidth=1)
    ax.invert_yaxis()
    ax.set_xlim(-1.02, 1.02)
    if i % n_cols != 0:
        ax.set_yticklabels([])

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.text(0.5, 0.03, 'Mean Pro Share Shift', ha='center')
fig.text(0.02, 0.5, 'Argument Case', va='center', rotation='vertical')
plt.tight_layout(rect=[0.04, 0.06, 1, 1])
plt.savefig('../figures/pro_share_shift_by_evidence_case.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# PAPER_ARTIFACT: effect_of_adding_one_piece_of_contradictory_evidence.pdf
apply_paper_style()

wide = stats_df.pivot(index='model', columns='case', values=['mean_pro_shift', 'ci_lower', 'ci_upper'])
model_order = [m for m in MODEL_ORDER if m in wide.index]

rows = []
for m in model_order:
    needed = ['pro', '75pro', 'con', '75con']
    if not all(c in wide['mean_pro_shift'].columns for c in needed):
        continue

    rows.append({
        'model': m,
        'track': 'pro',
        'start': wide.loc[m, ('mean_pro_shift', 'pro')],
        'end': wide.loc[m, ('mean_pro_shift', '75pro')],
        'start_low': wide.loc[m, ('ci_lower', 'pro')],
        'start_high': wide.loc[m, ('ci_upper', 'pro')],
        'end_low': wide.loc[m, ('ci_lower', '75pro')],
        'end_high': wide.loc[m, ('ci_upper', '75pro')],
    })
    rows.append({
        'model': m,
        'track': 'con',
        'start': wide.loc[m, ('mean_pro_shift', 'con')],
        'end': wide.loc[m, ('mean_pro_shift', '75con')],
        'start_low': wide.loc[m, ('ci_lower', 'con')],
        'start_high': wide.loc[m, ('ci_upper', 'con')],
        'end_low': wide.loc[m, ('ci_lower', '75con')],
        'end_high': wide.loc[m, ('ci_upper', '75con')],
    })

arrow_df = pd.DataFrame(rows)

# Order models by initial pro unanimity: highest pro mean first.
pro_unanimity = (
    arrow_df[arrow_df['track'] == 'pro']
    .loc[:, ['model', 'start']]
    .set_index('model')
    .sort_values('start', ascending=False)
)
plot_models = pro_unanimity.index.tolist()

fig, ax = plt.subplots(figsize=(16, 7))
y_base = {m: i for i, m in enumerate(plot_models)}
track_offset = {'pro': 0.16, 'con': -0.16}
track_color = {'pro': COLORS['pos'], 'con': COLORS['neg']}

for _, r in arrow_df.iterrows():
    y = y_base[r['model']] + track_offset[r['track']]
    color = track_color[r['track']]

    # CI lines are intentionally offset from the arrow so they are not read as arrow length.
    ci_start_y = y + 0.06
    ci_end_y = y - 0.06
    ax.plot([r['start_low'], r['start_high']], [ci_start_y, ci_start_y], color=color, alpha=0.45, linewidth=1.5, linestyle='--')
    ax.plot([r['end_low'], r['end_high']], [ci_end_y, ci_end_y], color=color, alpha=0.45, linewidth=1.5, linestyle=':')
    ax.plot([r['start'], r['start']], [y, ci_start_y], color=color, alpha=0.35, linewidth=1.0)
    ax.plot([r['end'], r['end']], [y, ci_end_y], color=color, alpha=0.35, linewidth=1.0)

    # Arrow from default to 75% case
    dx = r['end'] - r['start']
    ax.arrow(
        r['start'], y, dx, 0,
        length_includes_head=True,
        head_width=0.15,
        head_length=0.015,
        linewidth=2,
        color=color,
        alpha=0.9,
    )

    # Markers: hollow start, filled end
    ax.scatter(r['start'], y, s=45, facecolors='white', edgecolors=color, linewidths=1.5, zorder=3)
    ax.scatter(r['end'], y, s=45, facecolors=color, edgecolors=color, linewidths=1.0, zorder=3)

ax.axvline(0, color='black', linewidth=1)
ax.set_yticks([y_base[m] for m in plot_models])
ax.set_yticklabels(plot_models)
ax.set_xlabel('Mean Pro Share Shift vs Baseline')
ax.set_ylabel('Model')
ax.set_title('Shift from Default to 75% Contradictory-Evidence Cases')

legend_handles = [
    plt.Line2D([0], [0], color=COLORS['pos'], linewidth=2, marker='>', label='pro track: pro to 75pro'),
    plt.Line2D([0], [0], color=COLORS['neg'], linewidth=2, marker='>', label='con track: con to 75con'),
    plt.Line2D([0], [0], color='black', linewidth=0, marker='o', markerfacecolor='white', label='start (default)'),
    plt.Line2D([0], [0], color='black', linewidth=0, marker='o', markerfacecolor='black', label='end (75%)'),
    plt.Line2D([0], [0], color='black', linewidth=1.5, linestyle='--', label='start CI'),
    plt.Line2D([0], [0], color='black', linewidth=1.5, linestyle=':', label='end CI'),
]
ax.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)

x_min = min(arrow_df[['start_low', 'end_low']].min()) - 0.05
x_max = max(arrow_df[['start_high', 'end_high']].max()) + 0.05
ax.set_xlim(x_min, x_max)

# Reserve right margin for an external legend to avoid overlap.
plt.tight_layout(rect=[0, 0, 0.82, 1])
plt.savefig('../figures/effect_of_adding_one_piece_of_contradictory_evidence.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# PAPER_ARTIFACT: neither_all_models_opus_undecided.pdf
# PAPER_ARTIFACT: neither_all_models_gpt_grok_only.pdf
apply_paper_style()

# Build stance shares for baseline (neither) and balanced (all).
cases = ['neither', 'all']
stance_order = ['con', 'other', 'pro']
stance_colors = {'con': COLORS['neg'], 'other': COLORS['other'], 'pro': COLORS['pos']}

agg = (
    results_long[results_long['case'].isin(cases)]
    .groupby(['model', 'issue', 'case', 'issue_stance'])['count']
    .sum()
    .reset_index()
)
shares = agg.pivot_table(
    index=['model', 'issue', 'case'],
    columns='issue_stance',
    values='count',
    fill_value=0
).reset_index()
for s in stance_order:
    if s not in shares.columns:
        shares[s] = 0
shares['total'] = shares[stance_order].sum(axis=1).replace(0, np.nan)
for s in stance_order:
    shares[f'{s}_share'] = (shares[s] / shares['total']).fillna(0.0)


def _issue_slug(text: str) -> str:
    return text.lower().replace('&', 'and').replace(' ', '-').replace('--', '-')


def _issue_title(slug: str) -> str:
    return slug.replace('-', ' ').title()


def _binom_ci(k: float, n: float, conf: float = 0.95) -> tuple[float, float]:
    if n is None or np.isnan(n) or n <= 0:
        return (0.0, 0.0)
    k_i = int(round(float(k)))
    n_i = int(round(float(n)))
    if n_i <= 0:
        return (0.0, 0.0)
    ci = stats.binomtest(k_i, n_i).proportion_ci(confidence_level=conf, method='wilson')
    return (float(ci.low), float(ci.high))


def get_stance_metrics(model: str, issue: str, case: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    row = shares[(shares['model'] == model) & (shares['issue'] == issue) & (shares['case'] == case)]
    if row.empty:
        z = np.array([0.0, 0.0, 0.0])
        return z, z, z

    row = row.iloc[0]
    vals = np.array([float(row[f'{s}_share']) for s in stance_order], dtype=float)
    lows = []
    highs = []
    n = float(row['total']) if 'total' in row else np.nan
    for s in stance_order:
        low, high = _binom_ci(float(row[s]), n)
        lows.append(low)
        highs.append(high)
    return vals, np.array(lows, dtype=float), np.array(highs, dtype=float)


def get_other_decided_metrics(model: str, issue: str, case: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    row = shares[(shares['model'] == model) & (shares['issue'] == issue) & (shares['case'] == case)]
    if row.empty:
        z = np.array([0.0, 0.0])
        return z, z, z

    row = row.iloc[0]
    n = float(row['total']) if 'total' in row else np.nan
    other_k = float(row['other'])
    decided_k = float(row['pro']) + float(row['con'])

    other_share = float(row['other_share'])
    decided_share = float(row['pro_share']) + float(row['con_share'])

    other_low, other_high = _binom_ci(other_k, n)
    decided_low, decided_high = _binom_ci(decided_k, n)

    vals = np.array([other_share, decided_share], dtype=float)
    lows = np.array([other_low, decided_low], dtype=float)
    highs = np.array([other_high, decided_high], dtype=float)
    return vals, lows, highs


# -------- Opus-undecided issue panel (Opus only) --------
opus_model = 'claude-opus-4'
opus_issue_inputs = [
    'abortion', 'american-civil-liberties-union-ACLU', 'american-socialism', 'bill-clinton', 'constitutional-carry-of-guns',
    'dakota-access-pipeline', 'dc-and-puerto-rico-statehood', 'defund-the-police', 'drones',
    'gun-control', 'historical-statue-removal', 'immigration', 'minimum-wage', 'obamacare',
    'reparations-for-slavery', 'ronald-reagan', 'sanctuary-cities', 'universal-health-care',
    'wtc-muslim-center',
]
opus_issues = [_issue_slug(x) for x in opus_issue_inputs]

# Order issues by largest absolute shift in "other" share from baseline (neither) to balanced (all).
def _other_shift(issue: str) -> float:
    row_n = shares[(shares['model'] == opus_model) & (shares['issue'] == issue) & (shares['case'] == 'neither')]
    row_a = shares[(shares['model'] == opus_model) & (shares['issue'] == issue) & (shares['case'] == 'all')]
    if row_n.empty or row_a.empty:
        return 0.0
    return abs(float(row_a.iloc[0]['other_share']) - float(row_n.iloc[0]['other_share']))

opus_issues = sorted(opus_issues, key=_other_shift, reverse=True)

ncols = 4
nrows = int(np.ceil(len(opus_issues) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.0, nrows * 2.4), squeeze=False)

case_order = ['neither', 'all']
case_x = np.array([0, 1], dtype=float)
opus_categories = ['other', 'decided']
opus_offsets = {'other': -0.14, 'decided': 0.14}
opus_colors = {'other': stance_colors['other'], 'decided': COLORS['pos']}
bar_w = 0.25

for idx, issue in enumerate(opus_issues):
    r = idx // ncols
    c = idx % ncols
    ax = axes[r, c]

    metrics = {
        'neither': get_other_decided_metrics(opus_model, issue, 'neither'),
        'all': get_other_decided_metrics(opus_model, issue, 'all'),
    }

    for case_i, case_name in enumerate(case_order):
        vals, lows, highs = metrics[case_name]
        for s_idx, cat in enumerate(opus_categories):
            x = case_x[case_i] + opus_offsets[cat]
            y = vals[s_idx]
            yerr = [[max(0.0, y - lows[s_idx])], [max(0.0, highs[s_idx] - y)]]
            ax.bar(x, y, width=bar_w, color=opus_colors[cat], edgecolor='none')
            ax.errorbar(x, y, yerr=yerr, fmt='none', ecolor='black', elinewidth=0.9, capsize=2, zorder=4)

    # Per-issue significance stars for directional drop in Other (all < neither).
    row_n = shares[(shares['model'] == opus_model) & (shares['issue'] == issue) & (shares['case'] == 'neither')]
    row_a = shares[(shares['model'] == opus_model) & (shares['issue'] == issue) & (shares['case'] == 'all')]
    if not row_n.empty and not row_a.empty:
        row_n = row_n.iloc[0]
        row_a = row_a.iloc[0]
        n_n = int(row_n['total'])
        n_a = int(row_a['total'])

        other_drop = float(row_a['other_share']) < float(row_n['other_share'])
        if other_drop and n_n > 0 and n_a > 0:
            k_n = int(row_n['other'])
            k_a = int(row_a['other'])
            pval = stats.fisher_exact([[k_a, n_a - k_a], [k_n, n_n - k_n]], alternative='less').pvalue
            marker = p_to_marker(pval)
            if marker and marker != 'ns':
                s_idx = opus_categories.index('other')
                y_all_hi = metrics['all'][2][s_idx]
                x = case_x[1] + opus_offsets['other']
                ax.text(x, min(0.98, y_all_hi + 0.04), marker, ha='center', va='bottom', fontsize=14.4, fontweight='bold')

    ax.set_xlim(-0.55, 1.55)
    ax.set_ylim(0, 1.02)
    ax.set_title(_issue_title(issue), fontsize=17.6)
    ax.grid(axis='y', alpha=0.18, linewidth=0.6)
    ax.set_xticks(case_x)

    if r == nrows - 1:
        ax.set_xticklabels(['Neither', 'All'], fontsize=14.4)
    else:
        ax.set_xticklabels([])
        ax.tick_params(axis='x', bottom=False)

    if c == 0:
        ax.tick_params(axis='y', labelsize=9)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis='y', left=False)

for idx in range(len(opus_issues), nrows * ncols):
    axes[idx // ncols, idx % ncols].axis('off')

# Global paired significance across issues for Other-share drop (all vs neither).
paired_rows = []
for issue in opus_issues:
    row_n = shares[(shares['model'] == opus_model) & (shares['issue'] == issue) & (shares['case'] == 'neither')]
    row_a = shares[(shares['model'] == opus_model) & (shares['issue'] == issue) & (shares['case'] == 'all')]
    if row_n.empty or row_a.empty:
        continue
    paired_rows.append({
        'issue': issue,
        'other_neither': float(row_n.iloc[0]['other_share']),
        'other_all': float(row_a.iloc[0]['other_share']),
    })

paired_df = pd.DataFrame(paired_rows)
opus_p = np.nan
opus_marker = ''
opus_delta = np.nan
opus_ci_low = np.nan
opus_ci_high = np.nan
if len(paired_df) >= 2:
    d = paired_df['other_all'].to_numpy() - paired_df['other_neither'].to_numpy()
    opus_delta = float(np.mean(d))
    t_stat, opus_p = stats.ttest_rel(paired_df['other_all'], paired_df['other_neither'])
    sd = np.std(d, ddof=1)
    se = sd / np.sqrt(len(d)) if len(d) > 1 else 0.0
    t_crit = stats.t.ppf(0.975, df=len(d) - 1)
    half = float(t_crit * se)
    opus_ci_low = opus_delta - half
    opus_ci_high = opus_delta + half
    opus_marker = p_to_marker(opus_p)

opus_handles = [
    plt.Rectangle((0, 0), 1, 1, color=opus_colors['other']),
    plt.Rectangle((0, 0), 1, 1, color=opus_colors['decided']),
]
fig.legend(opus_handles, ['Other', 'Pro+Con'], loc='lower center', bbox_to_anchor=(0.5, 0.01), ncol=2, frameon=False)
fig.suptitle('ClaudeOpus 4: No Arguments vs. Balanced Arguments\n(Other vs Pro+Con)', y=0.98)
# if np.isfinite(opus_p):
#     fig.text(
#         0.5,
#         0.93,
#         # f"Paired across issues (Other share, all - neither): Δ={opus_delta:.3f} [{opus_ci_low:.3f}, {opus_ci_high:.3f}], p={opus_p:.3g} {opus_marker}",
#         ha='center',
#         va='center',
#         fontsize=16,
#     )
fig.subplots_adjust(hspace=0.60, wspace=0.30)
plt.tight_layout(rect=[0, 0.06, 1, 0.92])
plt.savefig('../figures/neither_all_models_opus_undecided.pdf', bbox_inches='tight')
plt.show()


# -------- GPT/Grok panel (all listed models per issue) --------
gg_models = [m for m in shares['model'].unique()]
gg_issue_inputs = [
    'binge-watching', 'drones', 'fighting-in-hockey', 'olympics', 'ronald-reagan',
    'saturday-halloween', 'school-uniforms', 'standardized-tests', 'teacher-tenure', 'us-penny',
]
gg_issues = [_issue_slug(x) for x in gg_issue_inputs]

ncols = 4
nrows = int(np.ceil(len(gg_issues) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.8, nrows * 3.6), squeeze=False)

bar_h = 0.50
for idx, issue in enumerate(gg_issues):
    r = idx // ncols
    c = idx % ncols
    ax = axes[r, c]
    y_base = np.arange(len(gg_models))

    for j, model in enumerate(gg_models):
        vals, _, _ = get_stance_metrics(model, issue, 'neither')  # Don't need CI values
        left = 0.0
        for s_idx, stance in enumerate(stance_order):
            width = vals[s_idx]
            ax.barh(
                y_base[j],
                width,
                left=left,
                color=stance_colors[stance],
                edgecolor='none',
                height=bar_h,
                alpha=1.0,
            )
            # CI bars removed
            left += width

    ax.set_xlim(0, 1)
    ax.set_title(_issue_title(issue), fontsize=17.6)
    ax.grid(axis='x', alpha=0.18, linewidth=0.6)

    if c == 0:
        ax.set_yticks(y_base)
        ax.set_yticklabels(gg_models, fontsize=16)
    else:
        ax.set_yticks(y_base)
        ax.set_yticklabels([])
        ax.tick_params(axis='y', left=False)

    if r == nrows - 1:
        ax.tick_params(axis='x', labelsize=9)
    else:
        ax.tick_params(axis='x', labelbottom=False, bottom=False)

for idx in range(len(gg_issues), nrows * ncols):
    axes[idx // ncols, idx % ncols].axis('off')

stance_handles = [plt.Rectangle((0, 0), 1, 1, color=stance_colors[s]) for s in stance_order]
fig.legend(
    stance_handles,
    ['Con', 'Other', 'Pro'],
    loc='lower center',
    bbox_to_anchor=(0.5, 0.01),
    ncol=3,
    frameon=False,
)
fig.suptitle('No Argument Stance Shares on Controversial Subset of Issues', y=0.98)
fig.subplots_adjust(hspace=0.58, wspace=0.30)
plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.savefig('../figures/neither_all_models_gpt_grok_only.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# PAPER_ARTIFACT: open_mindedness_by_topic_by_model.pdf
apply_paper_style()

# Expect open_mindedness_df from earlier cells; fallback to subset_open_mindedness_df
if 'open_mindedness_df' in globals():
    om_src = open_mindedness_df.copy()
elif 'subset_open_mindedness_df' in globals():
    om_src = subset_open_mindedness_df.copy()
else:
    raise RuntimeError('open_mindedness_df/subset_open_mindedness_df not found; run OM cells first.')

required_cols = {'model', 'issue', 'high_level_topic', 'open_mindedness_score'}
missing = required_cols - set(om_src.columns)
if missing:
    raise RuntimeError(f'Missing columns for OM topic figure: {missing}')

# Remove "Other" category from high_level_topic, if present
om_src = om_src[om_src['high_level_topic'].str.lower() != 'other']

# Dual uncertainty for OM:
# 1) trial-weighted CI (narrow, sampling uncertainty)
# 2) issue-bootstrap CI (wider, cross-issue heterogeneity)
from tqdm.auto import tqdm

om_cases = ['pro', '75pro', 'all', '75con', 'con']
om_weights = (
    results_long[results_long['case'].isin(om_cases)]
    .groupby(['model', 'issue'], as_index=False)['count']
    .sum()
    .rename(columns={'count': 'om_weight'})
)
om_src = om_src.merge(om_weights, on=['model', 'issue'], how='left')
om_src['om_weight'] = om_src['om_weight'].fillna(1.0).astype(float)


def summarize_dual_ci(g: pd.DataFrame, n_boot: int = 1200, seed: int = 42) -> dict:
    x = g['open_mindedness_score'].to_numpy(dtype=float)
    w = g['om_weight'].to_numpy(dtype=float)

    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x = x[mask]
    w = w[mask]
    if len(x) == 0:
        return {
            'mean': np.nan, 'count': 0,
            'trial_ci_low': np.nan, 'trial_ci_high': np.nan,
            'issue_ci_low': np.nan, 'issue_ci_high': np.nan,
        }

    mean_w = np.average(x, weights=w)
    if len(x) == 1:
        return {
            'mean': mean_w, 'count': 1,
            'trial_ci_low': mean_w, 'trial_ci_high': mean_w,
            'issue_ci_low': mean_w, 'issue_ci_high': mean_w,
        }

    # Trial-weighted CI
    var_w = np.average((x - mean_w) ** 2, weights=w)
    n_eff = (w.sum() ** 2) / np.sum(w ** 2)
    n_eff = max(float(n_eff), 1.0)
    sem_w = np.sqrt(var_w / n_eff) if n_eff > 0 else np.nan
    df_eff = max(n_eff - 1.0, 1.0)
    t_crit = stats.t.ppf(0.975, df_eff)
    ci_half_trial = t_crit * sem_w if np.isfinite(sem_w) else 0.0

    # Issue bootstrap CI (resample issues)
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot, dtype=float)
    n = len(x)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = np.average(x[idx], weights=w[idx])

    return {
        'mean': mean_w,
        'count': int(n),
        'trial_ci_low': float(mean_w - ci_half_trial),
        'trial_ci_high': float(mean_w + ci_half_trial),
        'issue_ci_low': float(np.quantile(boots, 0.025)),
        'issue_ci_high': float(np.quantile(boots, 0.975)),
    }


rows = []
group_items = list(om_src.groupby(['high_level_topic', 'model']))
for (topic, model), g in tqdm(group_items, desc='OM topic-model dual CI'):
    summary = summarize_dual_ci(g)
    rows.append({'high_level_topic': topic, 'model': model, **summary})

grouped = pd.DataFrame(rows)

# Remove "Other" from grouped DataFrame if present
grouped = grouped[grouped['high_level_topic'].str.lower() != 'other']

# Simplified visual: one dot color per model, grouped by topic, with horizontal CIs.
topic_order = grouped.groupby('high_level_topic')['mean'].max().sort_values(ascending=False).index.tolist()
model_order = [m for m in MODEL_ORDER if m in grouped['model'].unique() and 'grok' not in m and 'gpt' not in m]
if not model_order:
    model_order = sorted(grouped['model'].unique())

palette = sns.color_palette('tab10', n_colors=max(3, len(model_order)))
model_color = {m: palette[i % len(palette)] for i, m in enumerate(model_order)}

y_pos = {topic: i for i, topic in enumerate(topic_order)}
offsets = np.linspace(-0.28, 0.28, max(1, len(model_order)))
model_offset = {m: offsets[i] for i, m in enumerate(model_order)}

fig, ax = plt.subplots(figsize=(14, 0.62 * len(topic_order)))

for topic in topic_order:
    topic_data = grouped[grouped['high_level_topic'] == topic]
    y0 = y_pos[topic]
    if topic_data.empty:
        continue

    for _, row in topic_data.iterrows():
        model = row['model']
        if model not in model_offset:
            continue
        y = y0 + model_offset[model]
        x = float(row['mean'])
        c = model_color.get(model, 'gray')

        trial_low = float(row['trial_ci_low'])
        trial_high = float(row['trial_ci_high'])
        issue_low = float(row['issue_ci_low'])
        issue_high = float(row['issue_ci_high'])

        # Wider issue-level CI band (heterogeneity)
        ax.plot([issue_low, issue_high], [y, y], color=c, alpha=0.30, linewidth=4.0, solid_capstyle='round', zorder=1)
        # Narrower trial-level CI (sampling)
        ax.plot([trial_low, trial_high], [y, y], color=c, alpha=0.95, linewidth=1.4, zorder=2)
        ax.scatter(x, y, color=c, s=28, alpha=0.95, zorder=3)

ax.set_yticks(list(y_pos.values()))
ax.set_yticklabels(topic_order, fontsize=22.4)
ax.set_xlabel('Open-mindedness', fontsize=22.4)
ax.grid(axis='x', linestyle='--', alpha=0.5)
x_right = float(np.nanmax(grouped[['trial_ci_high', 'issue_ci_high']].to_numpy())) + 0.5
ax.set_xlim(left=0, right=max(8.0, x_right))

model_handles = [
    plt.Line2D([0], [0], marker='o', color=model_color[m], linestyle='', markersize=6, label=m)
    for m in model_order
]
ci_handles = [
    plt.Line2D([0], [0], color='black', linewidth=1.4, label='Trial-level 95% CI'),
    plt.Line2D([0], [0], color='black', linewidth=4.0, alpha=0.30, label='Issue-bootstrap 95% CI'),
]
legend1 = ax.legend(handles=model_handles, ncol=3, fontsize=14.4, frameon=False, loc='lower right', title='Model')
ax.add_artist(legend1)
ax.legend(handles=ci_handles, fontsize=14.4, frameon=False, loc='upper left')

plt.tight_layout()
plt.savefig('../figures/open_mindedness_by_topic_by_model.pdf', bbox_inches='tight')
plt.show()

grouped.sort_values(['high_level_topic', 'mean'], ascending=[True, False]).head(20)

In [ ]:
# PAPER_ARTIFACT: simple_evidence_correlation.pdf
apply_paper_style()

# Correlate evidence-induced shift magnitude with recency-alignment tendency.
# IMPORTANT: aggregate recency alignment by trial-weighted rate, not mean of subgroup percentages.
if 'agreed_with_last_count' in order_df.columns:
    order_agg = (
        order_df.groupby(['model', 'case'], as_index=False)
        .agg({'agreed_with_last_count': 'sum', 'total_trials': 'sum'})
    )
    order_agg['agreed_with_last_pct'] = (
        order_agg['agreed_with_last_count']
        / order_agg['total_trials'].replace(0, np.nan)
    )
else:
    # Backward-compatible fallback.
    order_agg = (
        order_df.groupby(['model', 'case'], as_index=False)
        .agg({'agreed_with_last_pct': 'mean', 'total_trials': 'sum'})
    )

merge_df = stats_df.merge(order_agg[['model', 'case', 'agreed_with_last_pct', 'total_trials']], on=['model', 'case'], how='inner')

# Include all one-sided and mixed cases.
case_order = ['con', '75con', 'all', '75pro', 'pro']
case_markers = {'con': 'v', '75con': 's', 'all': 'o', '75pro': 'D', 'pro': '^'}
case_labels = {'con': 'OS con', '75con': 'C&C con', 'all': 'Balanced', '75pro': 'C&C pro', 'pro': 'OS pro'}
case_colors = {'con': '#2E5AAC', '75con': '#5F7FC8', 'all': '#6E6E6E', '75pro': '#D58A3A', 'pro': '#C7661B'}

available_cases = [c for c in case_order if c in merge_df['case'].unique()]
merge_df = merge_df[merge_df['case'].isin(available_cases)].copy()
merge_df['case'] = pd.Categorical(merge_df['case'], categories=available_cases, ordered=True)

# Approx 95% CI for y-axis binomial proportion.
p = merge_df['agreed_with_last_pct']
n = merge_df['total_trials'].replace(0, np.nan)
se = np.sqrt((p * (1 - p)) / n)
merge_df['agree_ci_low'] = (p - 1.96 * se).clip(0, 1)
merge_df['agree_ci_high'] = (p + 1.96 * se).clip(0, 1)

# Group models into full-run vs partial-run areas.
full_run_models = [
    m for m in ['gemini-2.0-flash', 'claude-opus-4', 'claude-3.5-haiku', 'llama-3.1-405b', 'llama-3.1-8b']
    if m in merge_df['model'].unique()
]
partial_run_models = [
    m for m in ['gpt-4o', 'gpt-4o-mini', 'grok-3', 'grok-3-mini']
    if m in merge_df['model'].unique()
]

x_min = float(merge_df['ci_lower'].min()) - 0.05
x_max = float(merge_df['ci_upper'].max()) + 0.05
y_min = float(merge_df['agree_ci_low'].min()) - 0.02
y_max = float(merge_df['agree_ci_high'].max()) + 0.02

fig = plt.figure(figsize=(22, 8.5))
subfigs = fig.subfigures(2, 1, height_ratios=[1, 1])


def draw_model_group(subfig, model_list, group_title):
    n = max(1, len(model_list))
    axes = subfig.subplots(1, n, sharex=True, sharey=True)
    axes = np.atleast_1d(axes).tolist()

    for i, model in enumerate(model_list):
        ax = axes[i]
        g = merge_df[merge_df['model'] == model].sort_values('case')
        if g.empty:
            ax.axis('off')
            continue

        ax.plot(g['mean_pro_shift'], g['agreed_with_last_pct'], color='black', alpha=0.30, linewidth=1.2, zorder=1)

        for _, r in g.iterrows():
            c = case_colors.get(r['case'], '#666666')
            ax.errorbar(
                r['mean_pro_shift'],
                r['agreed_with_last_pct'],
                xerr=[[max(0.0, r['mean_pro_shift'] - r['ci_lower'])], [max(0.0, r['ci_upper'] - r['mean_pro_shift'])]],
                yerr=[[max(0.0, r['agreed_with_last_pct'] - r['agree_ci_low'])], [max(0.0, r['agree_ci_high'] - r['agreed_with_last_pct'])]],
                fmt=case_markers.get(r['case'], 'o'),
                color=c,
                ecolor=c,
                elinewidth=0.9,
                capsize=2,
                markersize=6,
                alpha=0.95,
                zorder=3,
            )

        ax.axhline(0.5, linestyle='--', color='black', linewidth=0.9)
        ax.axvline(0, linestyle='--', color='black', linewidth=0.9)
        ax.grid(alpha=0.20)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_title(model, fontsize=16)

    subfig.suptitle(group_title, fontsize=19.2, y=0.98)
    return axes


axes_top = draw_model_group(subfigs[0], full_run_models, 'Full-run models (Gemini / Claude / Llama)')
axes_bottom = draw_model_group(subfigs[1], partial_run_models, 'Partial-run models (GPT / Grok)')

# Axis labels on left-most panels and bottom row.
if len(axes_top) > 0:
    axes_top[0].set_ylabel('Agree-With-Last Proportion')
if len(axes_bottom) > 0:
    axes_bottom[0].set_ylabel('Agree-With-Last Proportion')
for ax in axes_bottom:
    ax.set_xlabel('Mean Pro Share Shift')

case_handles = [
    plt.Line2D([0], [0], marker=case_markers[c], color=case_colors[c], linestyle='', markersize=6, label=case_labels[c])
    for c in available_cases
]
fig.suptitle('Evidence Shift vs. Recency Alignment by Model Group', y=1.05, fontsize=22.4)
fig.legend(
    handles=case_handles,
    title='Case',
    ncol=len(available_cases),
    fontsize=16,
    title_fontsize=19.2,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.08),
    frameon=False,
)

plt.tight_layout(rect=[0.0, 0.14, 1.0, 0.94])
plt.savefig('../figures/simple_evidence_correlation.pdf', bbox_inches='tight')
plt.show()

In [ ]:
## Final paper tables with CI/significance

def with_significance(df, p_col='p_value'):
    out = df.copy()
    out['sig'] = out[p_col].apply(p_to_marker)
    return out

def fmt_val_sig_ci(val, sig, lo, hi, floatfmt="0.3f"):
    return f"${val:.3f}^{{{sig}}}$ $({lo:.3f},\\ {hi:.3f})$"

def pd_to_latex_table(
    df, columns, header_map=None, caption=None, label=None,
    index=False, col_format=None, escape=False, **kwargs
):
    """
    Print to_latex output, ready to copy-paste into LaTeX docs.
    """
    col_names = [header_map.get(col, col) if header_map else col for col in columns]
    df_out = df[columns].rename(columns={col: col_names[i] for i, col in enumerate(columns)})
    latex_str = df_out.to_latex(
        index=index,
        escape=escape,
        column_format=col_format or ("l" * len(columns)),
        caption=caption,
        label=label,
        **kwargs,
    )
    print(latex_str)

# table_counter_argument_shifts (from statistical summary deltas)
delta_tbl = stats_df[stats_df['case'].isin(['75pro', '75con'])].copy()
delta_tbl = delta_tbl[['model', 'case', 'mean_pro_shift', 'ci_lower', 'ci_upper', 'p_value']]
delta_tbl = with_significance(delta_tbl, 'p_value')
delta_tbl['shift_with_ci'] = delta_tbl.apply(
    lambda r: fmt_val_sig_ci(r['mean_pro_shift'], r['sig'], r['ci_lower'], r['ci_upper']), axis=1
)
print('PAPER_ARTIFACT: table_counter_argument_shifts')
delta_tbl_disp = delta_tbl.sort_values(['model', 'case'])[['model', 'case', 'shift_with_ci']]
pd_to_latex_table(
    delta_tbl_disp,
    columns=['model', 'case', 'shift_with_ci'],
    header_map={'model': 'Model', 'case': 'Case', 'shift_with_ci': 'Mean Shift$^{{sig}}$ (CI)'},
    caption="Counter-Argument Shifts with Significance and CIs.",
    label="tab:counter_argument_shifts",
    index=False,
    escape=False,
)

# table_om_full / table_om_subset with dual uncertainty:
# - Trial-level CI (sampling)
# - Issue-bootstrap CI (heterogeneity)
# - table_om_full excludes GPT/Grok
# - table_om_subset uses GG issue set


def trial_ci_by_model(df_in):
    src = df_in.copy()
    om_cases = ['pro', '75pro', 'all', '75con', 'con']
    om_weights = (
        results_long[results_long['case'].isin(om_cases)]
        .groupby(['model', 'issue'], as_index=False)['count']
        .sum()
        .rename(columns={'count': 'om_weight'})
    )
    src = src.merge(om_weights, on=['model', 'issue'], how='left')
    src['om_weight'] = src['om_weight'].fillna(1.0).astype(float)

    rows = []
    for model, g in src.groupby('model'):
        x = g['open_mindedness_score'].to_numpy(dtype=float)
        w = g['om_weight'].to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
        x = x[mask]
        w = w[mask]

        if len(x) == 0:
            rows.append({'model': model, 'om_mean': np.nan, 'trial_ci_low': np.nan, 'trial_ci_high': np.nan, 'trial_ci_pm': np.nan})
            continue

        mean_w = np.average(x, weights=w)
        if len(x) == 1:
            rows.append({'model': model, 'om_mean': mean_w, 'trial_ci_low': mean_w, 'trial_ci_high': mean_w, 'trial_ci_pm': 0.0})
            continue

        var_w = np.average((x - mean_w) ** 2, weights=w)
        n_eff = max((w.sum() ** 2) / np.sum(w ** 2), 1.0)
        sem_w = np.sqrt(var_w / n_eff) if n_eff > 0 else np.nan
        t_crit = stats.t.ppf(0.975, max(n_eff - 1.0, 1.0))
        ci_half_trial = float(t_crit * sem_w) if np.isfinite(sem_w) else 0.0

        rows.append({
            'model': model,
            'om_mean': mean_w,
            'trial_ci_low': mean_w - ci_half_trial,
            'trial_ci_high': mean_w + ci_half_trial,
            'trial_ci_pm': ci_half_trial,
        })

    return pd.DataFrame(rows)


if 'open_mindedness_df' in globals():
    om_all = open_mindedness_df.copy()

    # Full table excludes GPT/Grok families.
    om_full_src = om_all[~om_all['model'].str.contains('gpt|grok', case=False, regex=True)].copy()
    om_full_tbl = trial_ci_by_model(om_full_src)
    om_full_tbl['om_pm'] = om_full_tbl.apply(lambda r: f"${r['om_mean']:.3f} \\pm {r['trial_ci_pm']:.3f}$", axis=1)
    print('PAPER_ARTIFACT: table_om_full')
    om_full_disp = om_full_tbl.sort_values('om_mean', ascending=False)[['model', 'om_pm']]
    pd_to_latex_table(
        om_full_disp,
        columns=['model', 'om_pm'],
        header_map={'model': 'Model', 'om_pm': 'Open-Mindedness (mean $\\pm$ trial 95\\% CI half-width)'},
        caption='Open-mindedness by model (excludes GPT/Grok), reported as mean $\\pm$ trial 95\\% CI half-width.',
        label='tab:per-model-om-full',
        index=False,
        escape=False,
    )

    # Subset table is exactly the GG issue set across all models.
    gg_issue_inputs = [
        'binge-watching', 'drones', 'fighting-in-hockey', 'olympics', 'ronald-reagan',
        'saturday-halloween', 'school-uniforms', 'standardized-tests', 'teacher-tenure', 'us-penny',
    ]
    om_subset_src = om_all[om_all['issue'].isin(gg_issue_inputs)].copy()
    om_subset_tbl = trial_ci_by_model(om_subset_src)
    om_subset_tbl['om_pm'] = om_subset_tbl.apply(lambda r: f"${r['om_mean']:.3f} \\pm {r['trial_ci_pm']:.3f}$", axis=1)
    print('PAPER_ARTIFACT: table_om_subset')
    om_subset_disp = om_subset_tbl.sort_values('om_mean', ascending=False)[['model', 'om_pm']]
    pd_to_latex_table(
        om_subset_disp,
        columns=['model', 'om_pm'],
        header_map={'model': 'Model', 'om_pm': 'Open-Mindedness (mean $\\pm$ trial 95\\% CI half-width)'},
        caption='Open-mindedness on GG issue set (all models), reported as mean $\\pm$ trial 95\\% CI half-width.',
        label='tab:per-model-om-subset',
        index=False,
        escape=False,
    )

# table_baseline_pro_gt_con
baseline = results_long[results_long['case'] == 'neither'].copy()
base_agg = baseline.groupby(['model', 'issue', 'issue_stance'])['count'].sum().reset_index()
base_pivot = base_agg.pivot_table(index=['model', 'issue'], columns='issue_stance', values='count', fill_value=0).reset_index()
if 'pro' not in base_pivot.columns:
    base_pivot['pro'] = 0
if 'con' not in base_pivot.columns:
    base_pivot['con'] = 0
base_pivot['pro_gt_con'] = (base_pivot['pro'] > base_pivot['con']).astype(int)

# Model-level estimate and exact binomial significance vs chance (H0: p = 0.5).
binom_rows = []
for model, g in base_pivot.groupby('model'):
    k = int(g['pro_gt_con'].sum())
    n = int(len(g))
    p_hat = float(k / n) if n > 0 else np.nan
    if n > 0:
        p_val = float(stats.binomtest(k, n=n, p=0.5, alternative='two-sided').pvalue)
    else:
        p_val = np.nan
    binom_rows.append({'model': model, 'k_pro_gt_con': k, 'n_issues': n, 'p_hat': p_hat, 'p_value': p_val})

binom_tbl = pd.DataFrame(binom_rows)
if USE_BH_CORRECTION and not binom_tbl.empty:
    binom_tbl['p_value_adj'] = apply_bh_correction(binom_tbl['p_value'].values)
else:
    binom_tbl['p_value_adj'] = binom_tbl['p_value']

# Use adjusted p-values if BH is enabled; otherwise raw p-values.
p_col = 'p_value_adj' if USE_BH_CORRECTION else 'p_value'
tbl_pro_gt_con = with_significance(binom_tbl, p_col)
tbl_pro_gt_con['p_pro_gt_con'] = tbl_pro_gt_con.apply(
    lambda r: f"${r['p_hat']:.3f}^{{{r.get('sig', '')}}}$", axis=1
)
tbl_pro_gt_con['p_value_fmt'] = tbl_pro_gt_con[p_col].apply(lambda v: f"{v:.3g}" if np.isfinite(v) else 'nan')

print('PAPER_ARTIFACT: table_baseline_pro_gt_con')
tbl_pro_gt_con_disp = tbl_pro_gt_con.sort_values('p_hat', ascending=False)[['model', 'p_pro_gt_con', 'p_value_fmt', 'n_issues']]
pd_to_latex_table(
    tbl_pro_gt_con_disp,
    columns=['model', 'p_pro_gt_con', 'p_value_fmt', 'n_issues'],
    header_map={'model': 'Model', 'p_pro_gt_con': 'P(Pro $>$ Con)$^{sig}$', 'p_value_fmt': '$p$-value', 'n_issues': 'N issues'},
    caption='Baseline P(Pro $>$ Con) by model with exact binomial significance vs 0.5.',
    label='tab:baseline_pro_gt_con',
    index=False,
    escape=False,
)

# table_baseline_unanimous_consistent_disagreement_issue_counts
# Simple reproducible counts from baseline issue-level dispersion
group_issue = base_pivot.groupby('issue')['pro'].mean().reset_index(name='mean_pro')
# proxy disagreement: model variance in issue-level pro share
issue_var = base_pivot.groupby('issue')['pro'].var().reset_index(name='var_pro')
issue_stats = group_issue.merge(issue_var, on='issue', how='left').fillna(0)

summary_tbl = pd.DataFrame([
    {'category': 'Unanimous (low variance)', 'count': int((issue_stats['var_pro'] < issue_stats['var_pro'].quantile(0.2)).sum())},
    {'category': 'Unanimous and consistent', 'count': int(((issue_stats['var_pro'] < issue_stats['var_pro'].quantile(0.2)) & ((issue_stats['mean_pro'] > issue_stats['mean_pro'].quantile(0.8)) | (issue_stats['mean_pro'] < issue_stats['mean_pro'].quantile(0.2)))).sum())},
    {'category': 'Strong disagreement (high variance)', 'count': int((issue_stats['var_pro'] > issue_stats['var_pro'].quantile(0.8)).sum())},
])
print('PAPER_ARTIFACT: table_baseline_unanimous_consistent_disagreement_issue_counts')
pd_to_latex_table(
    summary_tbl,
    columns=['category', 'count'],
    header_map={'category': 'Category', 'count': 'Issue Count'},
    caption="Issue-level category counts: Unanimous, Consistent, and Disagreement.",
    label="tab:baseline_issue_counts",
    index=False,
    escape=False,
)

In [ ]:
## Validation: expected paper artifacts exist and are non-empty
from pathlib import Path

expected_files = [
    '../figures/pro_share_shift_by_evidence_case.pdf',
    '../figures/effect_of_adding_one_piece_of_contradictory_evidence.pdf',
    '../figures/neither_all_models_opus_undecided.pdf',
    '../figures/neither_all_models_gpt_grok_only.pdf',
    '../figures/open_mindedness_by_topic_by_model.pdf',
    '../figures/simple_evidence_correlation.pdf',
]

checks = []
for p in expected_files:
    path = Path(p)
    checks.append({'file': p, 'exists': path.exists(), 'size_bytes': path.stat().st_size if path.exists() else 0})

check_df = pd.DataFrame(checks)
display(check_df)

missing = check_df[~check_df['exists']]['file'].tolist()
empty = check_df[(check_df['exists']) & (check_df['size_bytes'] == 0)]['file'].tolist()
assert not missing, f'Missing generated files: {missing}'
assert not empty, f'Empty generated files: {empty}'
print('Paper artifact validation passed.')